# Event cross-sell: games & products for an event timeframe

Enter an **event or entertainment release name**. This notebook returns the catalog games and attach products that should be cross-sold during that window (lead-in → runtime → afterglow).

It builds on:

1. Notebook 01 — catalog + live calendar  
2. Notebook 02 — equivalent-event promotion plans  
3. Notebook 03 — daily trend priorities (optional context)

The same lookup powers the Floor Brief **Cross-sell** page in the web UI and desktop app.

Examples: `FIFA Women's World Cup`, `Gamescom`, `Spider-Man: Brand New Day`, `Roland-Garros`, `The Game Awards`.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.cross_sell import cross_sell_payload, format_cross_sell_table, unique_events
from src.load_data import load_adaptations, load_catalog, load_events
from src.promote import build_plans

events = load_events()
adaptations = load_adaptations()
catalog = load_catalog(games_only=True, drop_placeholder_dates=True)
plans = build_plans(events, adaptations, catalog)
print(f"promotion plans: {len(plans)}")
print(f"events with cross-sell SKUs: {len({p['event'] for p in plans})}")
print("sample upcoming events:")
for name in unique_events(plans, limit=12):
    print(" ·", name)


## Enter an event name

Change `EVENT_NAME` below and re-run the cell. Products are grouped by role (game hero, currency, DLC, edition).


In [ ]:
EVENT_NAME = "FIFA Women's World Cup"  # ← type any event / release name

# Resolve calendar row when possible (for runtime dates + related IP)
calendar = None
needle = EVENT_NAME.strip().lower()
for row in list(events) + list(adaptations):
    label = (row.get("event") or row.get("ip_adaptation") or "").strip()
    if label.lower() == needle or needle in label.lower():
        calendar = row
        break

payload = cross_sell_payload(EVENT_NAME, plans=plans, calendar_row=calendar)
print(format_cross_sell_table(payload))
print()
if payload.get("found"):
    print("Do this:", payload["do_this_today"]["headline"])
    for line in payload["in_short"]:
        print(" -", line)
    print("\nBy role:")
    for role, rows in (payload.get("by_role") or {}).items():
        titles = ", ".join(sorted({r["canonical_title"] for r in rows})[:8])
        print(f"  {role}: {titles}")


## Another example: theatrical / cross-media window

Spider-Man catalog titles that should be merchandised during the Brand New Day theatrical runtime.


In [ ]:
EVENT_NAME = "Spider-Man: Brand New Day"

calendar = next(
    (
        row
        for row in list(events) + list(adaptations)
        if (row.get("event") or row.get("ip_adaptation") or "") == EVENT_NAME
    ),
    None,
)
payload = cross_sell_payload(EVENT_NAME, plans=plans, calendar_row=calendar)
print(format_cross_sell_table(payload))
if payload.get("hero"):
    hero = payload["hero"]
    print(f"\nHero: {hero['canonical_title']} · offer: {hero.get('offer')}")
    print(hero.get("strategy_summary") or "")
    for phase in hero.get("phases") or []:
        print(f"  {phase['label']}: {phase['start']} → {phase['end']}")
        for tactic in (phase.get("tactics") or [])[:2]:
            print(f"    - {tactic}")


## Try any event interactively

Set `EVENT_NAME` to whatever you need (Gamescom, Roland-Garros, The Game Awards, …) and re-run.


In [ ]:
EVENT_NAME = "Gamescom"  # change me

calendar = next(
    (
        row
        for row in list(events) + list(adaptations)
        if EVENT_NAME.lower() in (row.get("event") or row.get("ip_adaptation") or "").lower()
    ),
    None,
)
payload = cross_sell_payload(EVENT_NAME, plans=plans, calendar_row=calendar)
print(format_cross_sell_table(payload))
